In [6]:
import numpy as np
import pandas as pd
import sys

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Default values (for Jupyter)
categories = ['sci.space', 'rec.autos', 'talk.politics.misc']
components = [50, 100, 200]

# If running from terminal, override using argparse
if "__file__" in globals():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--categories", nargs='+', required=True)
    parser.add_argument("--components", nargs='+', type=int, required=True)
    args = parser.parse_args()

    categories = args.categories
    components = args.components

print("Categories:", categories)
print("Components:", components)

Categories: ['sci.space', 'rec.autos', 'talk.politics.misc']
Components: [50, 100, 200]


In [9]:
def load_data(categories):
    print("Loading dataset...")
    data = fetch_20newsgroups(
        subset='all',
        categories=categories,
        remove=('headers', 'footers', 'quotes')
    )
    return data.data, data.target, data.target_names

documents, labels, target_names = load_data(categories)

Loading dataset...


In [10]:
def tfidf_vectorize(documents):
    print("Performing TF-IDF vectorization...")
    vectorizer = TfidfVectorizer(
        max_df=0.5,
        min_df=2,
        stop_words='english'
    )
    X = vectorizer.fit_transform(documents)
    return X, vectorizer

X, vectorizer = tfidf_vectorize(documents)

Performing TF-IDF vectorization...


In [11]:
def apply_svd(X, n_components):
    print(f"\nApplying TruncatedSVD with {n_components} components...")
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_reduced = svd.fit_transform(X)
    explained_variance = svd.explained_variance_ratio_.sum()
    return X_reduced, svd, explained_variance

In [12]:
def evaluate_clustering(X_reduced, n_clusters):
    print("Evaluating clustering...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_reduced)
    score = silhouette_score(X_reduced, labels)
    return score

In [13]:
def print_topics(svd, vectorizer, num_topics=5, num_terms=10):
    terms = vectorizer.get_feature_names_out()

    topic_lines = []
    topic_lines.append("Top Terms per Topic:\n")

    for i, comp in enumerate(svd.components_[:num_topics]):
        term_indices = np.argsort(comp)[::-1][:num_terms]
        topic_terms = [terms[idx] for idx in term_indices]

        line = f"Topic {i+1}: " + ", ".join(topic_terms)
        print(line)
        topic_lines.append(line)

    return topic_lines

In [14]:
results = []

for comp in components:
    X_reduced, svd, variance = apply_svd(X, comp)

    score = evaluate_clustering(X_reduced, len(categories))

    print(f"Explained Variance: {variance:.4f}")
    print(f"Silhouette Score: {score:.4f}")

    results.append([comp, variance, score])

    topic_lines = print_topics(svd, vectorizer)


Applying TruncatedSVD with 50 components...
Evaluating clustering...
Explained Variance: 0.1027
Silhouette Score: 0.0956
Topic 1: car, people, like, space, just, don, think, know, good, new
Topic 2: car, cars, engine, dealer, ford, oil, good, price, miles, buy
Topic 3: space, car, shuttle, nasa, launch, orbit, engine, mission, cars, thanks
Topic 4: men, gay, homosexual, sex, sexual, partners, homosexuals, male, promiscuous, study
Topic 5: insurance, health, car, private, space, tax, care, government, mail, president

Applying TruncatedSVD with 100 components...
Evaluating clustering...
Explained Variance: 0.1646
Silhouette Score: 0.0620
Topic 1: car, people, like, space, just, don, think, know, good, new
Topic 2: car, cars, engine, dealer, ford, oil, good, price, miles, buy
Topic 3: space, car, shuttle, nasa, launch, orbit, engine, mission, cars, thanks
Topic 4: men, gay, homosexual, sex, sexual, partners, homosexuals, male, promiscuous, study
Topic 5: insurance, health, car, space, p

In [15]:
df = pd.DataFrame(results, columns=["components", "explained_variance", "silhouette_score"])
df.to_csv("svd_results.csv", index=False)

with open("lsa_topic_terms.txt", "w") as f:
    for line in topic_lines:
        f.write(line + "\n")

print("\nFiles saved successfully!")


Files saved successfully!
